# PS S6E7 — N3: SWOT 소프트 보팅 앙상블 (HGB+TE / CatBoost / LGBM)

**설계 원칙 — 누수·과적합 제로**
- 모델 3종을 같은 파이프라인(규칙피처 + 수치형 exact-value TE, fold 내 교차적합)으로 학습
- SWOT 상보성 진단: 모델×결측세그먼트 성능 + 오답 상관
- 가중 소프트 보팅: 심플렉스 그리드 서치를 **중첩 검증**(OOF 행 5분할)으로 정직 평가
- **폴백 가드**: 정직 이득 < +0.0005 → HGB 솔로 채택 (V2 성능 하한 보장)
- 스태킹 아님 (메타모델 없음, 확률 가중평균만)

로컬 근거: V2(HGB+TE 5시드) OOF 0.95051 / LB 0.94982 · 오답의 85%가 결측 세그먼트에서 발생

In [ ]:
import numpy as np, pandas as pd, time, glob, os, itertools
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print('input mounts:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'NONE')
_c = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
assert _c, 'competition data not mounted'
DATA = os.path.dirname(_c[0]); print('DATA =', DATA)

CLASSES = ['at-risk','unhealthy','fit']; N_CLASS = 3; N_SPLITS = 5; FOLD_SEED = 42
train = pd.read_csv(f'{DATA}/train.csv'); test = pd.read_csv(f'{DATA}/test.csv')
mapping = {c:i for i,c in enumerate(CLASSES)}
y = train['health_condition'].map(mapping).to_numpy()
print(train.shape, test.shape, np.bincount(y))

In [ ]:
NUM = ['sleep_duration','heart_rate','bmi','calorie_expenditure',
       'step_count','exercise_duration','water_intake']
ORDINAL = {
    'stress_level':            {'low':0,'medium':1,'high':2},
    'sleep_quality':           {'poor':0,'average':1,'good':2},
    'physical_activity_level': {'sedentary':0,'moderate':1,'active':2},
    'smoking_alcohol':         {'no':0,'occasional':1,'yes':2},
    'diet_type':               {'veg':0,'balanced':1,'non-veg':2},
    'gender':                  {'female':0,'male':1,'other':2},
}

def encode(df):
    X = df[NUM + list(ORDINAL)].copy()
    for c, m in ORDINAL.items():
        X[c] = X[c].map(m).astype('float64')
    s, st, act = X['sleep_duration'], X['stress_level'], X['physical_activity_level']
    X['sleep_lt6'] = np.where(s.isna(), np.nan, (s < 6).astype(float))
    X['sleep_lt7'] = np.where(s.isna(), np.nan, (s < 7).astype(float))
    rp = np.full(len(X), np.nan)
    known = ~(s.isna() | st.isna() | act.isna())
    sv, stv, av = s[known], st[known], act[known]
    r = np.zeros(known.sum())
    r[(sv < 6) & (stv == 2)] = 1.0
    r[(sv >= 7) & (stv == 0) & (av == 2)] = 2.0
    rp[known.to_numpy()] = r
    X['rule_pred'] = rp
    X['miss_sleep'] = s.isna().astype(float)
    X['miss_stress'] = st.isna().astype(float)
    X['miss_activity'] = act.isna().astype(float)
    return X

X_all, X_test_all = encode(train), encode(test)
print(X_all.shape, X_test_all.shape)

In [ ]:
def make_model(kind, seed):
    if kind == 'hgb':
        return HistGradientBoostingClassifier(max_iter=2000, learning_rate=0.05,
            max_leaf_nodes=63, early_stopping=True, validation_fraction=0.08,
            n_iter_no_change=50, class_weight='balanced', random_state=seed)
    if kind == 'cat':
        return CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=8,
            auto_class_weights='Balanced', early_stopping_rounds=100, verbose=0,
            random_seed=seed, thread_count=-1)
    if kind == 'lgbm':
        return LGBMClassifier(n_estimators=1500, learning_rate=0.05, num_leaves=63,
            class_weight='balanced', verbose=-1, n_jobs=-1, random_state=seed)

def run_model(kind, seeds):
    oof = np.zeros((len(X_all), N_CLASS)); test_pred = np.zeros((len(X_test_all), N_CLASS))
    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=FOLD_SEED)  # 층화
        for fold, (tr, va) in enumerate(skf.split(X_all, y)):
            X_tr, X_va, X_te = X_all.iloc[tr].copy(), X_all.iloc[va].copy(), X_test_all.copy()
            y_tr = y[tr]
            te = TargetEncoder(target_type='multiclass', cv=5, random_state=seed)  # fold 내 교차적합
            cols = [f'te_{c}_{k}' for c in NUM for k in range(N_CLASS)]
            A = pd.DataFrame(te.fit_transform(X_tr[NUM].fillna(-999.0), y_tr), columns=cols)
            B = pd.DataFrame(te.transform(X_va[NUM].fillna(-999.0)), columns=cols)
            C = pd.DataFrame(te.transform(X_te[NUM].fillna(-999.0)), columns=cols)
            X_tr = pd.concat([X_tr.reset_index(drop=True), A], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), B], axis=1)
            X_te = pd.concat([X_te.reset_index(drop=True), C], axis=1)
            m = make_model(kind, seed + fold)
            m.fit(X_tr, y_tr)
            oof[va] += m.predict_proba(X_va) / len(seeds)
            test_pred += m.predict_proba(X_te) / (N_SPLITS * len(seeds))
    cv = balanced_accuracy_score(y, oof.argmax(1))
    print(f'[{kind} x{len(seeds)}seed] OOF={cv:.5f}')
    return oof, test_pred

t0 = time.time()
oofs, tests = {}, {}
oofs['hgb'],  tests['hgb']  = run_model('hgb',  [42, 2026, 7])
oofs['lgbm'], tests['lgbm'] = run_model('lgbm', [42])
oofs['cat'],  tests['cat']  = run_model('cat',  [42])
print(f'total train {time.time()-t0:.0f}s')

In [ ]:
# SWOT: 모델 x 결측세그먼트 성능 + 오답 상관 (상보성 진단)
segs = {
 'all_known': ~(train.sleep_duration.isna()|train.stress_level.isna()|train.physical_activity_level.isna()),
 'miss_sleep': train.sleep_duration.isna(), 'miss_stress': train.stress_level.isna(),
}
rows = []
for n, p in oofs.items():
    pred = p.argmax(1)
    r = {'model': n, 'overall': balanced_accuracy_score(y, pred)}
    for sn, m in segs.items():
        r[sn] = balanced_accuracy_score(y[m.to_numpy()], pred[m.to_numpy()])
    rows.append(r)
print(pd.DataFrame(rows).set_index('model').round(4).to_string())
names = list(oofs)
for a, b in itertools.combinations(names, 2):
    wa, wb = oofs[a].argmax(1) != y, oofs[b].argmax(1) != y
    print(f'{a} vs {b} | 오답겹침 {(wa&wb).sum()/max((wa|wb).sum(),1):.3f} | 예측일치 {(oofs[a].argmax(1)==oofs[b].argmax(1)).mean()*100:.2f}%')

In [ ]:
# 가중 소프트 보팅 — 심플렉스 그리드 + 중첩 검증 + 폴백 가드
P = np.stack([oofs[n] for n in names]); T = np.stack([tests[n] for n in names])
solo = {n: balanced_accuracy_score(y, oofs[n].argmax(1)) for n in names}
best_solo_name = max(solo, key=solo.get); best_solo = solo[best_solo_name]

grid = np.linspace(0, 1, 21)
combos = [w for w in itertools.product(grid, repeat=len(names)) if abs(sum(w)-1) < 1e-9]
kf = KFold(n_splits=5, shuffle=True, random_state=0)
honest, ws = [], []
for tr, va in kf.split(y):
    best_w, best_s = None, -1
    for w in combos:
        s = balanced_accuracy_score(y[tr], np.tensordot(np.array(w), P[:, tr], axes=1).argmax(1))
        if s > best_s: best_s, best_w = s, w
    honest.append(balanced_accuracy_score(y[va], np.tensordot(np.array(best_w), P[:, va], axes=1).argmax(1)))
    ws.append(best_w)
w_mean = np.mean(ws, axis=0); honest_gain = np.mean(honest) - best_solo
print('weights:', dict(zip(names, np.round(w_mean, 3))))
print(f'honest blend={np.mean(honest):.5f} | best solo({best_solo_name})={best_solo:.5f} | gain={honest_gain:+.5f}')

if honest_gain >= 0.0005:
    final_test = np.tensordot(w_mean, T, axes=1); choice = f'VOTING {dict(zip(names, np.round(w_mean,3)))}'
else:
    final_test = tests[best_solo_name]; choice = f'FALLBACK solo {best_solo_name} (gain {honest_gain:+.5f} < +0.0005)'
print('FINAL =', choice)

In [ ]:
inv = {i: c for i, c in enumerate(CLASSES)}
sub = pd.DataFrame({'id': test['id'], 'health_condition': [inv[i] for i in final_test.argmax(1)]})
sub.to_csv('submission.csv', index=False)
ss = pd.read_csv(f'{DATA}/sample_submission.csv')
assert len(sub) == len(ss) and list(sub.columns) == list(ss.columns)
assert sub['id'].tolist() == ss['id'].tolist() and set(sub['health_condition']) <= set(CLASSES)
print('submission.csv | sanity OK |', sub['health_condition'].value_counts().to_dict())

## 판정 로직 요약
- 정직 이득(중첩 검증) ≥ +0.0005 → 보팅 채택 / 미만 → 최고 솔로 폴백 (V2 하한 보장)
- 가중치가 한 모델에 90%+ 집중되면 상보성 없음의 증거 → 폴백과 사실상 동일
- 모든 튜닝은 OOF 내부에서만 (test 라벨 접촉 0, public LB 참조 0)